# Wildfire Case study 
 when the land actually breaks, does z break with it — in a way last year and day-of-year cannot explain?

### YEAR / ROLE 
2019 /// pre fire seasonal baseline 

2020 /// Event year (16 Aug 2020)

2021 /// Recovery 


In [ ]:
from pathlib import Path
import sys, os
from datetime import datetime, date 

import numpy as np
import pandas as pd 
import matplotlib.pyplot as plt
import torch 

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "src").exists():
    REPO_ROOT = Path.cwd().parent

sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)

from src.paths import (
    load_config, location_hls_dir, DATA_DIR, EMBEDDINGS_DIR, RESULTS_DIR,
)
from src.data_utils import download_hls_for_location, list_clear_observations
from src.prithvi_hls import index_hls_directory, lonlat_to_pixel, date_id_to_datetime, make_rgb_display
from src.prithvi_utils import load_single_frame_prithvi
from src.embedding_utils import (
    extract_single_date_embedding, save_embedding, embeddings_exist,
    load_location_sequence, build_metadata,
)
from src.similarity_utils import compare_token_pair, cosine_similarity_map
from src.visualization_utils import plot_change_map, save_fig
from src.ndvi_utils import ndvi_grids_for_location, ndvi_change_map, token_change_map, map_corr

CONFIG = load_config()
LOC = CONFIG["locations"]["location_09"]
SEARCH = CONFIG["search"]
LID = "location_09"

FIRE_DATE = date(2020, 8, 16)
T0, T1 = "2019-01-01", "2021-12-31"
# Ignition is 16 Aug 2020, but this 224px chip stays green through 2020-09-04
# (cosine vs pre ~0.99). First clear post-scar scene is 2020-10-14.
SCAR_ON_OR_AFTER = date(2020, 10, 1)

MIN_CLEAR = 0.50
DOWNLOAD_HLS = False
LOAD_PRITHVI = True
FORCE_RECOMPUTE = False 

HLS_DIR = location_hls_dir(LID, data_root=DATA_DIR)
EMB_ROOT = EMBEDDINGS_DIR
OUT = RESULTS_DIR / "project" / "fire_august_complex"
HLS_DIR.mkdir(parents=True, exist_ok=True)
(EMB_ROOT / LID).mkdir(parents=True, exist_ok=True)
OUT.mkdir(parents=True, exist_ok=True)
print(LOC["name"], LOC["latitude"], LOC["longitude"])
print("HLS", HLS_DIR)
print("Emb", EMB_ROOT / LID)

In [ ]:
if DOWNLOAD_HLS:
    download_hls_for_location(
        lon=LOC["longitude"],
        lat=LOC["latitude"],
        hls_dir=HLS_DIR,
        temporal_start=T0,
        temporal_end=T1,
        search_count=int(SEARCH.get("search_count", 20)),
        bbox_pad_deg=float(SEARCH.get("bbox_pad_deg", 0.05)),
        download=True,
        only_months=None,
        max_per_month=None,
    )
else :
    print ("skip downlload ")
    

In [ ]:
###  list clear dates and sanity check 

obs = index_hls_directory(HLS_DIR, lon=LOC["longitude"],lat=LOC["latitude"],patch_size=224)
sample = next(iter(obs.values()))
row, col, _, _ = lonlat_to_pixel(next(iter(sample.values())), LOC["longitude"], LOC["latitude"])

clear = list_clear_observations(obs, row, col, patch_size=224, min_clear_fraction=MIN_CLEAR)

print (f"{len(clear)} clear observations found")
for it in clear:
    print(it["date_key"], f"clear={it['clear_fraction']:.2f}")


In [ ]:
from src.prithvi_hls import (
    index_hls_directory, lonlat_to_pixel, date_id_to_datetime,
    make_rgb_display, load_observation,
)

In [ ]:
def item_date(it):
    return datetime.fromisoformat(it["date_key"]).date()

pre_list = [it for it in clear if item_date(it) < FIRE_DATE]
post_list = [it for it in clear if item_date(it) >= FIRE_DATE]
print("n pre", len(pre_list), "n post", len(post_list))
if not pre_list or not post_list:
    raise SystemExit("Need at least one clear date before and after the fire.")

pre = pre_list[-1]
post = next(it for it in post_list if item_date(it) >= SCAR_ON_OR_AFTER)
print("dates after ignition that are still pre-scar at this pin:")
for it in post_list:
    if item_date(it) < SCAR_ON_OR_AFTER:
        print(" ", it["date_key"], f"clear={it['clear_fraction']:.2f}")
print("pre ", pre["date_key"], "clear", f"{pre['clear_fraction']:.2f}")
print("post", post["date_key"], "clear", f"{post['clear_fraction']:.2f}")

def rgb_of(item):
    patch = load_observation(
        item["bands"], row=row, col=col, patch_size=224, apply_fmask=True,
    )
    # patch shape (6, 224, 224): B02, B03, B04, ...  make_rgb_display uses B04,B03,B02
    return make_rgb_display(patch)

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
for ax, item, label in (
    (axes[0], pre, "pre-fire"),
    (axes[1], post, "post-fire"),
):
    ax.imshow(rgb_of(item))
    ax.set_title(f"{label}\n{item['date_key']}  clear={item['clear_fraction']:.2f}")
    ax.axis("off")
plt.tight_layout()
fig.savefig(OUT / "rgb_pre_post.png", dpi=140, bbox_inches="tight")
plt.show()

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
bundle = load_single_frame_prithvi(repo_id=CONFIG["model"]["repo_id"], device=device)
model = bundle["model"]

for item in clear:
    key = item["date_key"]
    if embeddings_exist(LID,key,embeddings_root=EMB_ROOT) and not FORCE_RECOMPUTE:
        continue
    tokens,cls,raw = extract_single_date_embedding(
        model,device,item["bands"],row,col,LOC["longitude"],LOC["latitude"],
        item["date_id"],
    )
    meta = build_metadata(
        location_id=LID,
        location_name=LOC["name"],
        latitude=LOC["latitude"],
        longitude=LOC["longitude"],
        month_key=key,
        date_id=item["date_id"],
        clear_fraction=float(item["clear_fraction"]),
        model_name=CONFIG["model"]["repo_id"],
        device=str(device),
        bands_dict=item["bands"],
        center_row=row,
        center_col=col,
    )
    save_embedding(LID, key, tokens, cls, meta, embeddings_root=EMB_ROOT)
    print("saved", key, "clear", item["clear_fraction"])        

In [ ]:
seq = load_location_sequence(LID, embeddings_root=EMB_ROOT)

def item_date(it):
    return datetime.fromisoformat(it["key"][:10]).date()

seq = sorted(seq, key=item_date)
pre = [it for it in seq if item_date(it) < FIRE_DATE]
post = [it for it in seq if item_date(it) >= FIRE_DATE]
print("n pre", len(pre), "n post", len(post))

z_pre_item = pre[-1]
z_post_item = next(it for it in post if item_date(it) >= SCAR_ON_OR_AFTER)
print("z_pre", z_pre_item["key"], "z_post", z_post_item["key"])

In [ ]:
rows = []
z0 = z_pre_item["tokens"]
for it in seq:
    cmp_ = compare_token_pair(z0, it["tokens"], z_pre_item["cls"], it["cls"], n_shuffles=30)
    rows.append({
        "date": it["key"],
        "aligned": cmp_["aligned_mean"],
        "change": 1.0 - cmp_["aligned_mean"],
        "shuffle": cmp_["shuffle_mean"],
        "cls": cmp_.get("cls_cosine", np.nan),
        "after_fire": item_date(it) >= FIRE_DATE,
    })
ts = pd.DataFrame(rows)
ts.to_csv(OUT / "cosine_vs_prefire.csv", index=False)

fig, ax = plt.subplots(figsize=(10, 4))
dates = pd.to_datetime(ts["date"])
ax.plot(dates, ts["aligned"], marker="o", ms=4, label="aligned vs z_pre")
ax.plot(dates, ts["shuffle"], marker=".", ms=3, alpha=0.5, label="spatial shuffle")
ax.axvline(pd.Timestamp(FIRE_DATE), color="red", ls="--", label="ignition")
ax.axvline(pd.Timestamp(SCAR_ON_OR_AFTER), color="orange", ls=":", label="scar first visible")
ax.set_ylabel("cosine vs z_pre (2020-08-15)")
ax.set_ylim(0.75, 1.01)
ax.legend(loc="lower left")
fig.savefig(OUT / "cosine_vs_prefire.png", dpi=140, bbox_inches="tight")
plt.show()

key_dates = ["2020-08-15", "2020-08-20", "2020-09-04", "2020-10-14", "2021-08-15"]
print(ts.loc[ts["date"].isin(key_dates), ["date", "aligned", "change", "shuffle", "cls"]].to_string(index=False))
print("CLS never drops: min", float(ts["cls"].min()), "max", float(ts["cls"].max()))

In [ ]:
def nearest(seq, year, month, day=15):
    target = date(year, month, day)
    return min(seq, key=lambda it: abs((item_date(it) - target).days))

pairs = [
    ("2019 season only", nearest(seq, 2019, 7), nearest(seq, 2019, 10)),
    ("2020 fire", z_pre_item, z_post_item),
    ("2021 recovery", nearest(seq, 2021, 7), nearest(seq, 2021, 10)),
]
for name, a, b in pairs:
    c = compare_token_pair(a["tokens"], b["tokens"], n_shuffles=30)
    print(name, a["key"], "→", b["key"], "aligned", round(c["aligned_mean"], 3), "change", round(1-c["aligned_mean"], 3))

In [ ]:
post_doy = item_date(z_post_item).timetuple().tm_yday
cand_2019 = [it for it in seq if item_date(it).year == 2019]
seasonal = min(cand_2019, key=lambda it: abs(item_date(it).timetuple().tm_yday - post_doy))
c = compare_token_pair(seasonal["tokens"], z_post_item["tokens"], n_shuffles=30)
print("seasonal copy", seasonal["key"], "vs", z_post_item["key"],
      "aligned", round(c["aligned_mean"], 3), "change", round(1 - c["aligned_mean"], 3),
      "shuffle", round(c["shuffle_mean"], 3))

In [ ]:
from IPython.display import Image, display

cmap = 1.0 - cosine_similarity_map(z_pre_item["tokens"], z_post_item["tokens"])
c19 = 1.0 - cosine_similarity_map(pairs[0][1]["tokens"], pairs[0][2]["tokens"])
vmax = max(float(np.percentile(cmap, 98)), float(np.percentile(c19, 98)), 0.05)

plot_change_map(cmap, f"{z_pre_item['key']} → {z_post_item['key']}", OUT / "token_change_fire.png", vmax=vmax)
plot_change_map(c19, "2019 same-season gap", OUT / "token_change_2019.png", vmax=vmax)

print("fire  mean", round(float(cmap.mean()), 3), "p90", round(float(np.percentile(cmap, 90)), 3), "max", round(float(cmap.max()), 3))
print("2019  mean", round(float(c19.mean()), 3), "p90", round(float(np.percentile(c19, 90)), 3), "max", round(float(c19.max()), 3))
display(Image(filename=str(OUT / "token_change_fire.png")))
display(Image(filename=str(OUT / "token_change_2019.png")))